In [1]:
import pandas as pd

ROOT = r'/Users/Zac/PycharmProjects/Cartcounting_model_forecast'

# Decisions published by the ML server (comma-separated export of the
# receiver's scoring_results table).
data_alert = pd.read_csv(r'/Users/Zac/PycharmProjects/Cartcounting_model_forecast/cart_platform/outputs/experiment.csv')

data_alert['decision_time'] = pd.to_datetime(data_alert['decision_time'])

# Raw telemetry is tab-separated with a BOM.
data_raw = pd.read_csv(r'/Users/Zac/PycharmProjects/Cartcounting_model_forecast/cart_platform/data/real_data.csv', sep='\t')
data_raw['DVRDate'] = pd.to_datetime(data_raw['DVRDate'])
data_raw['next_DVRDate'] = pd.to_datetime(data_raw['next_DVRDate'])
print(data_alert.shape, data_raw.shape)

(2523, 21) (2543, 12)


In [9]:
data_alert[data_alert['alert_state'].isin([
    'CRITICAL'])]

,id,request_id,entity_id,kdvr,chan_no,site_key,decision_time,current_cart,simulated_cart,pred_cart_20min,...,pred_min_20min,pred_min_40min,lower_bound_min_40min,shortage_risk,alert_state,quantity,model_id,calibration_id,scored_at,urgency
1057,11723,req-2119405238,163:215:0,215,0,163,2026-10-04 12:48:03,6.0,6.0,6.0,...,3.23,2.79,-4.03,0.7311,CRITICAL,0,mdl_163_215_0_20261016025457_174a61,cal_163_215_0_7f0d4fec,2026-10-16T02:56:45.310+00:00,NONE
1062,11728,req-2119405606,163:215:0,215,0,163,2026-10-04 13:21:25,5.0,8.5,5.0,...,3.74,2.64,-4.18,0.7470,CRITICAL,0,mdl_163_215_0_20261016025457_174a61,cal_163_215_0_7f0d4fec,2026-10-16T02:56:45.678+00:00,NONE
1063,11729,req-2119405680,163:215:0,215,0,163,2026-10-04 13:28:06,9.0,9.0,9.0,...,2.71,1.14,-5.68,0.8566,CRITICAL,20,mdl_163_215_0_20261016025457_174a61,cal_163_215_0_7f0d4fec,2026-10-16T02:56:45.788+00:00,URGENT
1065,11731,req-2119405869,163:215:0,215,0,163,2026-10-04 13:44:47,4.0,4.0,4.0,...,2.46,0.52,-6.30,0.8805,CRITICAL,0,mdl_163_215_0_20261016025457_174a61,cal_163_215_0_7f0d4fec,2026-10-16T02:56:45.956+00:00,NONE
1066,11732,req-2119405958,163:215:0,215,0,163,2026-10-04 13:51:27,3.0,11.0,3.0,...,3.76,2.34,-4.48,0.7709,CRITICAL,0,mdl_163_215_0_20261016025457_174a61,cal_163_215_0_7f0d4fec,2026-10-16T02:56:46.032+00:00,NONE
1993,12659,req-2119471220,163:215:0,215,0,163,2026-10-09 12:06:15,3.0,3.0,3.0,...,2.92,2.91,-3.91,0.7191,CRITICAL,0,mdl_163_215_0_20261016025457_174a61,cal_163_215_0_7f0d4fec,2026-10-16T02:57:51.287+00:00,NONE


In [3]:
data_alert.groupby('alert_state')['shortage_risk'].describe()

,count,mean,std,min,25%,50%,75%,max
alert_state,,,,,,,,
CRITICAL,6.0,0.784200,0.068020,0.7191,0.735075,0.75895,0.835175,0.8805
NORMAL,106.0,0.006974,0.011022,0.0000,0.000000,0.00000,0.012000,0.0657
WARNING,2411.0,0.008681,0.044129,0.0000,0.000000,0.00000,0.000000,0.6534


In [16]:
# import pandas as pd
# import numpy as np
#
# # ============================================================
# # CONFIG
# # ============================================================
#
# WARNING_THRESHOLD = 0.0686
# CRITICAL_THRESHOLD = 0.3888
# COOLDOWN_MINUTES = 0
#
# # ============================================================
# # PREPROCESSING
# # ============================================================
#
# data_alert = data_alert.copy()
#
# data_alert['decision_time'] = pd.to_datetime(
#     data_alert['decision_time'],
#     errors='coerce'
# )
#
# data_alert['shortage_risk'] = pd.to_numeric(
#     data_alert['shortage_risk'],
#     errors='coerce'
# )
#
# data_alert = (
#     data_alert
#     .sort_values('decision_time')
#     .reset_index(drop=True)
# )
#
# # ============================================================
# # STEP 1: CLASSIFY ALERT BASED ON SHORTAGE RISK
# # ============================================================
#
# data_alert['alert_state'] = np.select(
#     [
#         data_alert['shortage_risk'] > CRITICAL_THRESHOLD,
#         data_alert['shortage_risk'] > WARNING_THRESHOLD
#     ],
#     [
#         'CRITICAL',
#         'WARNING'
#     ],
#     default='NORMAL'
# )
#
# # ============================================================
# # STEP 2: APPLY INDEPENDENT COOLDOWN
# # ============================================================
#
# cooldown = pd.Timedelta(minutes=COOLDOWN_MINUTES)
#
# last_alert_time = {
#     'WARNING': None,
#     'CRITICAL': None
# }
#
# for idx, row in data_alert.iterrows():
#
#     state = row['alert_state']
#     current_time = row['decision_time']
#
#     # NORMAL remains NORMAL
#     if state == 'NORMAL':
#         continue
#
#     # Invalid timestamp: do not emit alert
#     if pd.isna(current_time):
#         data_alert.at[idx, 'alert_state'] = 'NORMAL'
#         continue
#
#     previous_time = last_alert_time[state]
#
#     if (
#         previous_time is None
#         or current_time - previous_time >= cooldown
#     ):
#         # Allow alert and update cooldown timestamp
#         last_alert_time[state] = current_time
#
#     else:
#         # Suppress repeated alert of the same type
#         data_alert.at[idx, 'alert_state'] = 'NORMAL'
#
# # ============================================================
# # RESULT
# # ============================================================
#
# print(
#     data_alert[
#         ['decision_time', 'shortage_risk', 'alert_state']
#     ].head(30)
# )

In [17]:
import numpy as np

target_capacity = 28
bay_capacity = 60
model_mae = 3.583447

data_alert['quantity'] = np.minimum(
    np.maximum(
        0,
        np.ceil(
            target_capacity
            - data_alert['pred_min_40min']
            +  model_mae
        )
    ),
    np.maximum(
        0,
        np.floor(bay_capacity - data_alert['current_cart'])
    )
).astype(int)

In [4]:
final_result = data_raw.merge(

    data_alert[['decision_time','alert_state','shortage_risk','quantity','urgency']],
# should_replenish
    how='left',

    left_on='next_DVRDate',

    right_on='decision_time'

)
final_result = final_result[final_result['alert_state'].notnull()]

In [5]:
final_result['mmdd'] = final_result['decision_time'].dt.strftime('%m-%d')


In [6]:
final_result.groupby('mmdd')['alert_state'].value_counts().unstack()

alert_state,CRITICAL,NORMAL,WARNING
mmdd,,,
09-29,NaN,106.0,65.0
09-30,NaN,NaN,192.0
10-01,NaN,NaN,196.0
10-02,NaN,NaN,202.0
10-03,NaN,NaN,197.0
10-04,5.0,NaN,190.0
10-05,NaN,NaN,188.0
10-06,NaN,NaN,184.0
10-07,NaN,NaN,188.0


In [24]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 1. DATA PREPROCESSING
# ============================================================

df = final_result.copy()

df['decision_time'] = pd.to_datetime(
    df['decision_time'],
    errors='coerce'
)

numeric_cols = [
    'Count',
    'Count_IN',
    'should_replenish',
    'shortage_risk',
    'quantity'
]

for col in numeric_cols:
    if col not in df.columns:
        if col == 'Count_IN':
            raise ValueError("final_result must contain 'Count_IN'")
        df[col] = None

    df[col] = pd.to_numeric(df[col], errors='coerce')

df['should_replenish'] = df['should_replenish'].fillna(0)

df['alert_state'] = (
    df['alert_state']
    .fillna('NORMAL')
    .astype(str)
    .str.upper()
)

df = (
    df.dropna(subset=['decision_time'])
    .sort_values('decision_time')
    .reset_index(drop=True)
)

# ============================================================
# 2. CREATE FIGURE - TWO CHARTS, SHARED X AXIS
# ============================================================

fig = make_subplots(
    rows=2,
    cols=1,

    shared_xaxes=True,

    vertical_spacing=0.08,

    row_heights=[0.7, 0.3],

    specs=[
        [{"secondary_y": True}],
        [{"secondary_y": False}]
    ],

    subplot_titles=[
        'Cart Availability & Predicted Shortage Risk',
        'Customer Traffic IN'
    ]
)

# ============================================================
# 3. CHART 1 - ACTUAL CART COUNT
# ============================================================

fig.add_trace(
    go.Scatter(
        x=df['decision_time'],
        y=df['Count'],

        mode='lines',

        name='Actual Cart Count',

        line=dict(
            color='#64748B',
            width=2
        ),

        opacity=0.85,

        hovertemplate=(
            '<b>Time:</b> %{x}<br>'
            '<b>Cart Count:</b> %{y:.0f}'
            '<extra></extra>'
        )
    ),

    row=1,
    col=1,
    secondary_y=False
)

# ============================================================
# 4. CHART 1 - SHORTAGE RISK PROBABILITY
# ============================================================
#
# fig.add_trace(
#     go.Scatter(
#         x=df['decision_time'],
#         y=df['shortage_risk'],
#
#         mode='lines',
#
#         name='Predicted Shortage Risk',
#
#         line=dict(
#             color='#2563EB',
#             width=2.5
#         ),
#
#         opacity=0.9,
#
#         hovertemplate=(
#             '<b>Time:</b> %{x}<br>'
#             '<b>Shortage Risk:</b> %{y:.1%}'
#             '<extra></extra>'
#         )
#     ),
#
#     row=1,
#     col=1,
#     secondary_y=True
# )

# ============================================================
# 5. CHART 1 - ALERT MARKERS
# ============================================================

colors = {
    'WARNING': '#F1C40F',
    'CRITICAL': '#E74C3C'
}

for state, color in colors.items():

    subset = df[df['alert_state'] == state]

    for replenish_flag in [0, 1]:

        points = subset[
            subset['should_replenish'] == replenish_flag
        ]

        if points.empty:
            continue

        marker_size = 7 if replenish_flag == 0 else 17
        marker_line_width = 0 if replenish_flag == 0 else 2

        label = (
            state
            if replenish_flag == 0
            else f'{state} + Replenishment'
        )

        customdata = points[
            [
                'shortage_risk',
                'quantity',
                'should_replenish'
            ]
        ].to_numpy()

        fig.add_trace(
            go.Scatter(
                x=points['decision_time'],
                y=points['Count'],

                mode='markers',

                name=label,

                marker=dict(
                    color=color,
                    size=marker_size,
                    line=dict(
                        color='black',
                        width=marker_line_width
                    )
                ),

                customdata=customdata,

                hovertemplate=(
                    '<b>Time:</b> %{x}<br>'
                    '<b>Cart Count:</b> %{y:.0f}<br>'
                    f'<b>Alert State:</b> {state}<br>'
                    '<b>Shortage Risk:</b> %{customdata[0]:.1%}<br>'
                    '<b>Recommended Quantity:</b> %{customdata[1]}<br>'
                    '<b>Should Replenish:</b> %{customdata[2]}'
                    '<extra></extra>'
                )
            ),

            row=1,
            col=1,
            secondary_y=False
        )

# ============================================================
# 6. CHART 1 - CART THRESHOLDS
# ============================================================

fig.add_hline(
    y=20,
    line_dash='dash',
    line_color='#F59E0B',
    line_width=1.5,
    annotation_text='AMBER (20 carts)',
    annotation_position='top left',
    row=1,
    col=1,
    secondary_y=False
)

fig.add_hline(
    y=5,
    line_dash='dash',
    line_color='#DC2626',
    line_width=1.5,
    annotation_text='RED (5 carts)',
    annotation_position='bottom left',
    row=1,
    col=1,
    secondary_y=False
)

# ============================================================
# 7. CHART 1 - SHORTAGE RISK THRESHOLD
# ============================================================

RISK_THRESHOLD = 0.20

fig.add_hline(
    y=RISK_THRESHOLD,
    line_dash='dot',
    line_color='#2563EB',
    line_width=1.5,
    annotation_text=f'Risk Threshold ({RISK_THRESHOLD:.0%})',
    annotation_position='top right',
    row=1,
    col=1,
    secondary_y=True
)

# ============================================================
# 8. CHART 2 - CUSTOMER TRAFFIC IN
# ============================================================

fig.add_trace(
    go.Bar(
        x=df['decision_time'],
        y=df['Count_IN'],

        name='Customer Traffic IN',

        marker=dict(
            color='#10B981',
            opacity=0.75
        ),

        hovertemplate=(
            '<b>Time:</b> %{x}<br>'
            '<b>Traffic IN:</b> %{y:.0f}'
            '<extra></extra>'
        )
    ),

    row=2,
    col=1
)

# ============================================================
# 9. LAYOUT
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            'Cart Availability, Shortage Risk, '
            'Predictive Alerts & Customer Traffic'
        ),
        font=dict(size=22)
    ),

    template='plotly_white',

    height=1000,

    hovermode='x unified',

    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0
    ),

    margin=dict(
        l=80,
        r=90,
        t=150,
        b=80
    ),

    bargap=0.1
)

# ============================================================
# 10. X AXES
# ============================================================

fig.update_xaxes(
    showgrid=True,
    gridcolor='#E5E7EB',

    tickformat='%m-%d %H:%M',

    rangeselector=dict(
        buttons=[
            dict(count=1, label='1D', step='day', stepmode='backward'),
            dict(count=3, label='3D', step='day', stepmode='backward'),
            dict(count=7, label='7D', step='day', stepmode='backward'),
            dict(step='all', label='ALL')
        ]
    ),

    row=1,
    col=1
)

fig.update_xaxes(
    title_text='Decision Time',

    rangeslider=dict(
        visible=True,
        thickness=0.08
    ),

    row=2,
    col=1
)

# ============================================================
# 11. Y AXES
# ============================================================

# Chart 1 - Left Y axis
fig.update_yaxes(
    title_text='Available Cart Count',
    rangemode='tozero',
    showgrid=True,
    gridcolor='#E5E7EB',
    row=1,
    col=1,
    secondary_y=False
)

# Chart 1 - Right Y axis
fig.update_yaxes(
    title_text='Shortage Probability',
    range=[0, 1],
    tickformat='.0%',
    dtick=0.1,
    showgrid=False,
    row=1,
    col=1,
    secondary_y=True
)

# Chart 2 - Traffic IN
fig.update_yaxes(
    title_text='Customer Traffic IN',
    rangemode='tozero',
    showgrid=True,
    gridcolor='#E5E7EB',
    row=2,
    col=1
)

# ============================================================
# 12. EXPORT HTML
# ============================================================

output_file = 'cart_alert_dashboard.html'

fig.write_html(
    output_file,
    include_plotlyjs=True,
    full_html=True
)

print(f'HTML saved: {output_file}')

fig.show()

HTML saved: cart_alert_dashboard.html


In [ ]:
# data_alert = data_alert.sort_values('decision_time').copy()
#
# COOLDOWN = pd.Timedelta(minutes=40)
#
# RISK_THRESHOLD = 0.15
#
# last_warning_time = None
#
# for idx, row in data_alert.iterrows():
#
#     # Keep existing CRITICAL alerts
#
#     if row['alert_state'] == 'CRITICAL':
#
#         continue
#
#     current_time = row['decision_time']
#
#     risk = row['shortage_risk']
#
#     # Default state
#
#     data_alert.at[idx, 'alert_state'] = 'NORMAL'
#
#     if pd.notna(risk) and risk > RISK_THRESHOLD:
#
#         # First warning or cooldown expired
#
#         if (
#
#             last_warning_time is None
#
#             or current_time - last_warning_time >= COOLDOWN
#
#         ):
#
#             data_alert.at[idx, 'alert_state'] = 'WARNING'
#
#             last_warning_time = current_time

In [7]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 1. CONFIGURATION
# ============================================================

AMBER_THRESHOLD = 20
RED_THRESHOLD = 5

URGENCY_COLORS = {
    "LOW": "#22C55E",
    "MEDIUM": "#F59E0B",
    "HIGH": "#F97316",
    "URGENT": "#DC2626"
}

URGENCY_ORDER = ["LOW", "MEDIUM", "HIGH", "URGENT"]

# ============================================================
# 2. DATA PREPROCESSING
# ============================================================

df = final_result.copy()

df["decision_time"] = pd.to_datetime(
    df["decision_time"],
    errors="coerce"
)

numeric_cols = [
    "Count",
    "Count_IN",
    "should_replenish",
    "shortage_risk",
    "quantity"
]

for col in numeric_cols:

    if col not in df.columns:
        if col == "Count_IN":
            raise ValueError(
                "final_result must contain 'Count_IN'"
            )

        df[col] = None

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df["should_replenish"] = (
    df["should_replenish"].fillna(0)
)

df["quantity"] = (
    df["quantity"].fillna(0)
)

if "alert_state" not in df.columns:
    df["alert_state"] = "NORMAL"

df["alert_state"] = (
    df["alert_state"]
    .fillna("NORMAL")
    .astype(str)
    .str.upper()
)

# Urgency preprocessing
if "urgency" not in df.columns:
    df["urgency"] = "NONE"

df["urgency"] = (
    df["urgency"]
    .fillna("NONE")
    .astype(str)
    .str.upper()
    .str.strip()
)

df = (
    df.dropna(subset=["decision_time"])
    .sort_values("decision_time")
    .reset_index(drop=True)
)

# ============================================================
# 3. FILTER ALERT MARKERS
# ============================================================

# WARNING: only when quantity > 0
warning_points = df[
    (df["alert_state"] == "WARNING") &
    (df["quantity"] > 0)
].copy()

# CRITICAL: show all
critical_points = df[
    df["alert_state"] == "CRITICAL"
].copy()

# Replenishment orders
replenishment_points = df[
    df["quantity"] > 0
].copy()

print("=" * 60)
print("CART ALERT SUMMARY")
print("=" * 60)

print(f"Total records: {len(df)}")
print(
    f"WARNING with quantity > 0: "
    f"{len(warning_points)}"
)
print(f"CRITICAL: {len(critical_points)}")
print(
    f"Replenishment decisions (quantity > 0): "
    f"{len(replenishment_points)}"
)

print("\nURGENCY DISTRIBUTION")

print(
    replenishment_points["urgency"]
    .value_counts()
    .reindex(URGENCY_ORDER, fill_value=0)
)

# ============================================================
# 4. CREATE FIGURE
# ============================================================

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.7, 0.3],
    specs=[
        [{"secondary_y": True}],
        [{"secondary_y": False}]
    ],
    subplot_titles=[
        "Cart Availability & Predictive Alerts",
        "Customer Traffic IN"
    ]
)

# ============================================================
# 5. ACTUAL CART COUNT
# ============================================================

fig.add_trace(
    go.Scatter(
        x=df["decision_time"],
        y=df["Count"],
        mode="lines",
        name="Actual Cart Count",
        line=dict(
            color="#64748B",
            width=2
        ),
        opacity=0.85,
        hovertemplate=(
            "<b>Time:</b> %{x}<br>"
            "<b>Cart Count:</b> %{y:.0f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1,
    secondary_y=False
)

# ============================================================
# 6. WARNING MARKERS
# ONLY QUANTITY > 0
# ============================================================

if not warning_points.empty:

    warning_customdata = warning_points[
        [
            "shortage_risk",
            "quantity",
            "should_replenish",
            "urgency"
        ]
    ].to_numpy()

    fig.add_trace(
        go.Scatter(
            x=warning_points["decision_time"],
            y=warning_points["Count"],
            mode="markers",
            name="WARNING (Quantity > 0)",
            marker=dict(
                color="#F1C40F",
                size=13,
                symbol="circle",
                line=dict(
                    color="black",
                    width=1.5
                )
            ),
            customdata=warning_customdata,
            hovertemplate=(
                "<b>Time:</b> %{x}<br>"
                "<b>Cart Count:</b> %{y:.0f}<br>"
                "<b>Alert State:</b> WARNING<br>"
                "<b>Shortage Risk:</b> "
                "%{customdata[0]:.1%}<br>"
                "<b>Recommended Quantity:</b> "
                "%{customdata[1]:.0f}<br>"
                "<b>Should Replenish:</b> "
                "%{customdata[2]:.0f}<br>"
                "<b>Urgency:</b> "
                "%{customdata[3]}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=1,
        secondary_y=False
    )

# ============================================================
# 7. CRITICAL MARKERS
# SHOW ALL
# ============================================================

if not critical_points.empty:

    critical_customdata = critical_points[
        [
            "shortage_risk",
            "quantity",
            "should_replenish",
            "urgency"
        ]
    ].to_numpy()

    fig.add_trace(
        go.Scatter(
            x=critical_points["decision_time"],
            y=critical_points["Count"],
            mode="markers",
            name="CRITICAL",
            marker=dict(
                color="#E74C3C",
                size=15,
                symbol="circle",
                line=dict(
                    color="black",
                    width=1.5
                )
            ),
            customdata=critical_customdata,
            hovertemplate=(
                "<b>Time:</b> %{x}<br>"
                "<b>Cart Count:</b> %{y:.0f}<br>"
                "<b>Alert State:</b> CRITICAL<br>"
                "<b>Shortage Risk:</b> "
                "%{customdata[0]:.1%}<br>"
                "<b>Recommended Quantity:</b> "
                "%{customdata[1]:.0f}<br>"
                "<b>Should Replenish:</b> "
                "%{customdata[2]:.0f}<br>"
                "<b>Urgency:</b> "
                "%{customdata[3]}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=1,
        secondary_y=False
    )

# ============================================================
# 8. URGENCY MARKERS
#
# Show urgency as DIAMOND markers.
# Only rows with quantity > 0.
#
# Offset slightly above cart line so urgency markers
# do not completely overlap WARNING / CRITICAL markers.
# ============================================================

for urgency in URGENCY_ORDER:

    urgency_points = replenishment_points[
        replenishment_points["urgency"] == urgency
    ].copy()

    if urgency_points.empty:
        continue

    urgency_customdata = urgency_points[
        [
            "Count",
            "shortage_risk",
            "quantity",
            "alert_state",
            "should_replenish"
        ]
    ].to_numpy()

    fig.add_trace(
        go.Scatter(
            x=urgency_points["decision_time"],

            # Visual offset for readability
            y=urgency_points["Count"] + 3,

            mode="markers",
            name=f"Urgency: {urgency}",

            marker=dict(
                color=URGENCY_COLORS[urgency],
                size=11,
                symbol="diamond",
                line=dict(
                    color="white",
                    width=1.5
                )
            ),

            customdata=urgency_customdata,

            hovertemplate=(
                "<b>Time:</b> %{x}<br>"
                "<b>Urgency:</b> " + urgency + "<br>"
                "<b>Actual Cart Count:</b> "
                "%{customdata[0]:.0f}<br>"
                "<b>Shortage Risk:</b> "
                "%{customdata[1]:.1%}<br>"
                "<b>Quantity:</b> "
                "%{customdata[2]:.0f}<br>"
                "<b>Alert State:</b> "
                "%{customdata[3]}<br>"
                "<b>Should Replenish:</b> "
                "%{customdata[4]:.0f}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=1,
        secondary_y=False
    )

# ============================================================
# 9. CART THRESHOLDS
# ============================================================

fig.add_hline(
    y=AMBER_THRESHOLD,
    line_dash="dash",
    line_color="#F59E0B",
    line_width=1.5,
    annotation_text=(
        f"AMBER ({AMBER_THRESHOLD} carts)"
    ),
    annotation_position="top left",
    row=1,
    col=1,
    secondary_y=False
)

fig.add_hline(
    y=RED_THRESHOLD,
    line_dash="dash",
    line_color="#DC2626",
    line_width=1.5,
    annotation_text=(
        f"RED ({RED_THRESHOLD} carts)"
    ),
    annotation_position="bottom left",
    row=1,
    col=1,
    secondary_y=False
)

# ============================================================
# 10. CUSTOMER TRAFFIC IN
# ============================================================

fig.add_trace(
    go.Bar(
        x=df["decision_time"],
        y=df["Count_IN"],
        name="Customer Traffic IN",
        marker=dict(
            color="#10B981",
            opacity=0.75
        ),
        hovertemplate=(
            "<b>Time:</b> %{x}<br>"
            "<b>Traffic IN:</b> %{y:.0f}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=1
)

# ============================================================
# 11. LAYOUT
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Cart Availability, Predictive Alerts, "
            "Replenishment Urgency & Customer Traffic"
        ),
        font=dict(size=22)
    ),
    template="plotly_white",
    height=1000,
    hovermode="x unified",

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    ),

    margin=dict(
        l=80,
        r=90,
        t=150,
        b=80
    ),

    bargap=0.1
)

# ============================================================
# 12. X AXES
# ============================================================

fig.update_xaxes(
    showgrid=True,
    gridcolor="#E5E7EB",
    tickformat="%m-%d %H:%M",

    rangeselector=dict(
        buttons=[
            dict(
                count=1,
                label="1D",
                step="day",
                stepmode="backward"
            ),
            dict(
                count=3,
                label="3D",
                step="day",
                stepmode="backward"
            ),
            dict(
                count=7,
                label="7D",
                step="day",
                stepmode="backward"
            ),
            dict(
                step="all",
                label="ALL"
            )
        ]
    ),

    row=1,
    col=1
)

fig.update_xaxes(
    title_text="Decision Time",
    rangeslider=dict(
        visible=True,
        thickness=0.08
    ),
    row=2,
    col=1
)

# ============================================================
# 13. Y AXES
# ============================================================

fig.update_yaxes(
    title_text="Available Cart Count",
    rangemode="tozero",
    showgrid=True,
    gridcolor="#E5E7EB",
    row=1,
    col=1,
    secondary_y=False
)

fig.update_yaxes(
    title_text="Shortage Probability",
    range=[0, 1],
    tickformat=".0%",
    dtick=0.1,
    showgrid=False,
    row=1,
    col=1,
    secondary_y=True
)

fig.update_yaxes(
    title_text="Customer Traffic IN",
    rangemode="tozero",
    showgrid=True,
    gridcolor="#E5E7EB",
    row=2,
    col=1
)

# ============================================================
# 14. EXPORT HTML
# ============================================================

output_file = "cart_alert_urgency_dashboard.html"

fig.write_html(
    output_file,
    include_plotlyjs=True,
    full_html=True
)

print(f"\nHTML saved: {output_file}")

fig.show()

CART ALERT SUMMARY
Total records: 2523
WARNING with quantity > 0: 18
CRITICAL: 6
Replenishment decisions (quantity > 0): 19

URGENCY DISTRIBUTION
urgency
LOW        0
MEDIUM    13
HIGH       1
URGENT     5
Name: count, dtype: int64

HTML saved: cart_alert_urgency_dashboard.html
